# 02 — Random Forest: Detection, Classification & Ablation
Train scikit-learn Random Forest models for:
1. **Binary fault detection** — healthy vs. faulted
2. **Multi-class fault classification** — which fault type?
3. **Feature ablation** — how much do SOLEY physics features help?
4. **Temporal robustness** — generalise to unseen years
5. **Cross-location generalisation** — generalise to unseen sites


## 1. Imports & setup

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from library.utils         import setup_logging
from library.config        import BatchConfig
from library.data          import load_data_rf
from library.features      import add_features
from library.models        import train_rf_detection, train_rf_classification
from library.evaluation    import (run_temporal_split, run_cross_location,
                                    run_feature_ablation)
from library.visualization import plot_confusion, plot_importance

setup_logging()
%matplotlib inline
plt.rcParams["figure.dpi"] = 120


## 2. Configuration

In [2]:
DATA_DIR   = "data"
OUTPUT_DIR = "outputs/random_forest"
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

cfg = BatchConfig(DATA_DIR)
print(cfg)


  Config: array=5.34 kWp, 6 location(s)
  God-mode columns found in data and excluded from all feature sets: ['detailed_balance_efficiency_pct', 'ff', 'jmpp_a_m2', 'jsc_a_m2', 'vmpp_v', 'voc_v']
  Columns: 11 SCADA, 8 device physics, 6 stress, 29 constant (excluded), 6 god-mode (excluded), 25 total features
BatchConfig(array_kwp=5.34, locations=6, features=25)


## 3. Load & engineer features

In [3]:
# Full feature set: SCADA + device physics + stress indicators
feature_list = cfg.feature_set("full")   # or "scada", "scada+stress"

df = load_data_rf(DATA_DIR, cfg, max_rows=2_000_000)
print(f"Dataset shape: {df.shape}")
print(f"Features used: {len(feature_list)}")


Loading 310 files (12 fault types) …
Balanced subsampling: ~166,666 rows per fault type
After daytime filter: 7,819 rows
After daytime filter: 7,786 rows
After daytime filter: 7,820 rows
After daytime filter: 7,829 rows
After daytime filter: 7,818 rows
After daytime filter: 7,832 rows
After daytime filter: 7,829 rows
After daytime filter: 7,814 rows
After daytime filter: 7,794 rows
After daytime filter: 7,795 rows
After daytime filter: 7,804 rows
After daytime filter: 7,786 rows
After daytime filter: 7,744 rows
After daytime filter: 7,728 rows
After daytime filter: 7,734 rows
After daytime filter: 7,632 rows
After daytime filter: 7,653 rows
After daytime filter: 7,655 rows
After daytime filter: 7,800 rows
After daytime filter: 7,801 rows
After daytime filter: 7,809 rows
After daytime filter: 9,098 rows
After daytime filter: 9,101 rows
After daytime filter: 9,095 rows
After daytime filter: 9,122 rows
  … loaded 25 / 310 files
After daytime filter: 9,116 rows
After daytime filter: 9,130 

## 4. Task 1 — Binary Fault Detection

In [ ]:
model_det, auc_det, cv_det = train_rf_detection(
    df, feature_list, cfg,
    save_path=f"{OUTPUT_DIR}/rf_model_detection.pkl",
)
print(f"\nHold-out ROC AUC: {auc_det:.4f}")
print(f"5-fold CV AUC:    {cv_det.mean():.4f} ± {cv_det.std():.4f}")



  RF FAULT DETECTION (binary)
  Healthy: 648,249  (33.1%)
  Faulted: 1,310,872  (66.9%)
  Training RF (200 trees) …
  Trained in 238.7s

              precision    recall  f1-score   support

     Healthy       0.72      0.91      0.80    129650
     Faulted       0.95      0.82      0.88    262175

    accuracy                           0.85    391825
   macro avg       0.83      0.87      0.84    391825
weighted avg       0.87      0.85      0.85    391825

  ROC AUC (hold-out): 0.9285
  Running 5-fold CV …


In [ ]:
# Feature importance
plot_importance(
    model_det, feature_list,
    "Fault Detection",
    f"{OUTPUT_DIR}/importance_detection.png",
    scada_features=cfg.scada_features,
    device_features=cfg.device_features,
    top_n=20,
)
from IPython.display import Image
Image(f"{OUTPUT_DIR}/importance_detection.png")


## 5. Task 2 — Multi-class Fault Classification

In [ ]:
model_cls, auc_cls, cv_cls, le = train_rf_classification(
    df, feature_list, cfg,
    save_path=f"{OUTPUT_DIR}/rf_model_classification.pkl",
    save_deployment_path=f"{OUTPUT_DIR}/rf_deployment_classification.pkl",
)
if auc_cls:
    print(f"\nHold-out Weighted ROC AUC: {auc_cls:.4f}")
    if cv_cls is not None:
        print(f"5-fold CV AUC:             {cv_cls.mean():.4f} ± {cv_cls.std():.4f}")


In [ ]:
plot_importance(
    model_cls, feature_list,
    "Fault Classification",
    f"{OUTPUT_DIR}/importance_classification.png",
    scada_features=cfg.scada_features,
    device_features=cfg.device_features,
)
Image(f"{OUTPUT_DIR}/importance_classification.png")


## 6. Task 3 — Temporal Robustness

In [ ]:
temporal_results = run_temporal_split(df, feature_list, pathlib.Path(OUTPUT_DIR), cfg)
print("\nTemporal split results:")
for k, v in temporal_results.items():
    print(f"  {k}: {v:.4f}")


## 7. Task 4 — Cross-Location Generalisation

In [ ]:
# Requires data from multiple sites — skip gracefully if only one site
cross_results = run_cross_location(df, feature_list, pathlib.Path(OUTPUT_DIR), cfg)
if cross_results:
    import pandas as pd
    display(pd.DataFrame(cross_results).set_index("test_location"))


## 8. Task 5 — Feature Ablation

In [ ]:
ablation_results = run_feature_ablation(df, pathlib.Path(OUTPUT_DIR), cfg)
import pandas as pd
display(pd.DataFrame(ablation_results))


In [ ]:
Image(f"{OUTPUT_DIR}/feature_ablation.png")


## 9. Results Summary

In [ ]:
print("=" * 55)
print("RANDOM FOREST RESULTS SUMMARY")
print("=" * 55)
print(f"Detection     ROC AUC (hold-out): {auc_det:.4f}")
print(f"Detection     ROC AUC (5-fold CV): {cv_det.mean():.4f} ± {cv_det.std():.4f}")
if auc_cls:
    print(f"Classification ROC AUC (hold-out): {auc_cls:.4f}")
for k, v in temporal_results.items():
    print(f"Temporal {k}:  {v:.4f}")
if cross_results:
    import numpy as np
    avg_d = np.nanmean([r['detection_auc'] for r in cross_results])
    avg_c = np.nanmean([r['classification_auc'] for r in cross_results])
    print(f"Cross-location avg Detection AUC:       {avg_d:.4f}")
    print(f"Cross-location avg Classification AUC: {avg_c:.4f}")
delta = ablation_results[-1]['auc'] - ablation_results[0]['auc']
print(f"Ablation delta (Full - SCADA only):  {delta:+.4f} AUC")
print(f"\nAll outputs saved to: {OUTPUT_DIR}")
